[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB06_evaluation_leakage.ipynb)

# NB06 · Evaluate honestly: ROC, PR, threshold, calibration, leakage
**Artificial Intelligence in Medical Imaging** · Day 3 · Lab 7 (40 min)

| | |
|---|---|
| Runtime | CPU is enough (the CNN trains in about a minute) |
| Data | PneumoniaMNIST 28 px, 4,708 / 524 / 624, CC BY 4.0 (same data as Day 2) |
| Checkpoints | AUC by hand equals scikit-learn · validation sensitivity ≥ 0.90 at the chosen threshold · **the test set is evaluated exactly once** |
| Tiers | **Core** = ROC/PR, threshold, one test evaluation · **Plus** = bootstrap interval, calibration, leakage demo · **Challenge** = temperature scaling |

**Learning goals.** (1) Judge a classifier with ROC and precision–recall curves, not accuracy. (2) Choose a decision threshold from a clinical requirement, on the validation set. (3) Touch the test set once. (4) Put an error bar on every number and check whether probabilities can be trusted.

**The one-shot rule.** One cell touches the test set. The notebook logs how many times it ran. Run it **once**, after the threshold is frozen.

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, time, math
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
for d in ("data", "outputs"): Path(d).mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

In [ ]:
def load_pneumonia():
    """PneumoniaMNIST (28 px, CC BY 4.0). Falls back to SYNTHETIC data if the download fails."""
    try:
        from medmnist import PneumoniaMNIST
        out = {}
        for s in ("train", "val", "test"):
            d = PneumoniaMNIST(split=s, download=True, root="data")
            out[s] = (d.imgs.reshape(len(d.imgs), 28, 28).astype(np.uint8), d.labels.reshape(-1).astype(np.int64))
        return out, True
    except Exception as e:
        print("!! Could not download PneumoniaMNIST:", repr(e))
        print("!! Using SYNTHETIC data so the code still runs. Numbers below are NOT about pneumonia.")
        rng = np.random.default_rng(0); out = {}
        for s, n in (("train", 2000), ("val", 300), ("test", 300)):
            y = (rng.random(n) < 0.7).astype(np.int64)
            x = rng.normal(100, 25, (n, 28, 28)); x[y == 1, 8:20, 8:20] += 40
            out[s] = (x.clip(0, 255).astype(np.uint8), y)
        return out, False

data, REAL = load_pneumonia()
to_tensor = lambda x: ((torch.from_numpy(x).float() / 255.0 - 0.5) / 0.5).unsqueeze(1)    # (N, 1, 28, 28), range -1..1
X = {s: to_tensor(v[0]) for s, v in data.items()}
Y = {s: torch.from_numpy(v[1]) for s, v in data.items()}
print({s: tuple(X[s].shape) for s in X}, "| real data:", REAL)
maj_val = max(Y["val"].float().mean().item(), 1 - Y["val"].float().mean().item())
print(f"majority-class rate on validation: {maj_val:.3f}  (an 'always the same answer' model scores this)")

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
import torch.nn.functional as F

def make_loader(split, bs=128, shuffle=False, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(X[split], Y[split]), batch_size=bs, shuffle=shuffle, generator=g if shuffle else None)

def build_cnn():       # the 2,050-parameter CNN of Day 2
    return nn.Sequential(nn.Conv2d(1, 8, 3), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(8, 16, 3), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(400, 2))

def fit(model, epochs=8, lr=1e-3, seed=SEED):
    """Same recipe as NB04. Uses the TRAINING set only (the validation set is for choices, the test set is untouched)."""
    set_seed(seed); model.to(device); loader = make_loader("train", shuffle=True, seed=seed)
    crit = nn.CrossEntropyLoss(); opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    return model

@torch.no_grad()
def get_scores(model, split):
    """Return P(pneumonia), the binary logit z = z1 - z0, and the true labels. Eval mode, no gradients."""
    model.eval(); P, Z = [], []
    for xb, _ in make_loader(split, bs=512):
        out = model(xb.to(device)).cpu(); Z.append(out[:, 1] - out[:, 0]); P.append(torch.softmax(out, 1)[:, 1])
    return torch.cat(P).numpy(), torch.cat(Z).numpy(), Y[split].numpy()
print("helpers ready")

## 1. The model to judge
We retrain the Day-2 CNN with the same recipe and seed (about a minute). We only look at **validation** scores until the threshold is frozen.

In [ ]:
model = fit(build_cnn(), epochs=8)
p_val, z_val, y_val = get_scores(model, "val")
print("validation images:", len(y_val), "| share of pneumonia:", round(float(y_val.mean()), 3))
print("score range:", p_val.min().round(3), "to", p_val.max().round(3))

### AUC by hand
AUC has a plain meaning: **the probability that a randomly chosen pneumonia image gets a higher score than a randomly chosen normal image** (ties count half). Compare *every* pneumonia score with *every* normal score.

**TODO 1.** Write `auc_by_hand(scores, labels)`. Hint: `scores[labels==1][:, None] > scores[labels==0][None, :]` compares all pairs at once.

In [ ]:
def auc_by_hand(scores, labels):
    pos, neg = scores[labels == 1], scores[labels == 0]
    # TODO 1: fraction of (positive, negative) pairs where the positive scores higher; add half of the ties
    return 0.0

In [ ]:
auc_val = roc_auc_score(y_val, p_val)
print(f"AUC (scikit-learn) {auc_val:.4f} | AUC by hand {auc_by_hand(p_val, y_val):.4f}")
assert abs(auc_by_hand(p_val, y_val) - auc_val) < 1e-9, "your AUC differs from scikit-learn: check the pairs and the ties"
assert auc_val > 0.5, "the model should rank better than chance"
rng = np.random.default_rng(0); toy_s = rng.random(50); toy_y = (rng.random(50) > 0.5).astype(int)
assert abs(auc_by_hand(toy_s, toy_y) - roc_auc_score(toy_y, toy_s)) < 1e-9
print("Checkpoint 1 passed: AUC by hand = scikit-learn")

## 2. ROC and precision–recall on the validation set

In [ ]:
fpr, tpr, thr = roc_curve(y_val, p_val)
prec, rec, thr_pr = precision_recall_curve(y_val, p_val)
ap = average_precision_score(y_val, p_val); prevalence = y_val.mean()
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "k:"); ax[0].set_xlabel("1 - specificity (false positive rate)"); ax[0].set_ylabel("sensitivity"); ax[0].set_title(f"ROC, AUC {auc_val:.3f}")
ax[1].plot(rec, prec); ax[1].axhline(prevalence, color="grey", ls=":"); ax[1].set_xlabel("recall (= sensitivity)"); ax[1].set_ylabel("precision"); ax[1].set_title(f"precision-recall, AP {ap:.3f}")
plt.tight_layout(); plt.show()
print(f"prevalence of pneumonia in validation: {prevalence:.2f} (dotted line in the PR plot = a model with no skill)")

> **Read the two plots.** ROC ignores how common the disease is. The precision–recall curve does not: at low prevalence, precision collapses even when the AUC looks fine. Here pneumonia is common (see the prevalence), so the two views are more alike than in a screening setting.

## 3. Choose the threshold from the clinical requirement
**Requirement (stated by the clinician):** *the tool may miss at most 10% of pneumonia cases*, i.e. **sensitivity ≥ 0.90**. Among all thresholds that meet it, pick the one with the **highest specificity** (fewest false alarms).

Work on the **validation** set. **Never** use 0.5 just because it is the default, and never pick the threshold on the test set.

**TODO 2.** Using `fpr`, `tpr`, `thr` from `roc_curve`, find the first point (highest threshold) with `tpr >= REQUIRED_SENSITIVITY`. Hint: `np.argmax(tpr >= REQUIRED_SENSITIVITY)`.

In [ ]:
REQUIRED_SENSITIVITY = 0.90
# TODO 2a: index of the first ROC point with tpr >= REQUIRED_SENSITIVITY
idx = 0
# TODO 2b: the threshold at that index
threshold = float(thr[idx])
print(f"threshold {threshold:.3f} -> validation sensitivity {tpr[idx]:.3f}, specificity {1 - fpr[idx]:.3f}")

### Checkpoint 2 · the requirement holds on validation

In [ ]:
pred_val = (p_val >= threshold).astype(int)
tp = int(((pred_val == 1) & (y_val == 1)).sum()); fn = int(((pred_val == 0) & (y_val == 1)).sum())
tn = int(((pred_val == 0) & (y_val == 0)).sum()); fp = int(((pred_val == 1) & (y_val == 0)).sum())
sens, spec = tp / (tp + fn), tn / (tn + fp)
print(f"validation: sensitivity {sens:.3f} | specificity {spec:.3f} | false alarms {fp} of {tn + fp} normal images")
assert sens >= REQUIRED_SENSITIVITY, "the chosen threshold misses more than 10% of pneumonia on validation"
assert 0 < threshold <= 1 and spec > 0, "the threshold should be a probability and keep some specificity"
print("Checkpoint 2 passed")

> **Say the cost out loud.** A requirement of 90% sensitivity has a price: the specificity above. That price (false alarms, extra tests) is the real result, and the clinician has to accept it.

## 4. Freeze everything, then touch the test set once
Before the next cell, write down (here, in the notebook) what you will report: the model, the threshold, the metrics. After this cell you may **not** change any of them.

The test cell is wrapped in a function that counts its calls. It cannot stop you from cheating; it makes cheating visible.

In [ ]:
FROZEN = {"model": "CNN of Day 2, seed 42, 8 epochs", "threshold": threshold, "requirement": f"sensitivity >= {REQUIRED_SENSITIVITY} on validation", "report": ["AUC", "sensitivity", "specificity", "precision"]}
print(FROZEN)

_test_runs = []
def final_test_report():
    """THE ONLY CELL THAT TOUCHES THE TEST SET. Run it once."""
    _test_runs.append(1)
    if len(_test_runs) > 1: print(f"!! WARNING: the test set has now been used {len(_test_runs)} times. It is no longer a test set.")
    p_t, z_t, y_t = get_scores(model, "test")
    pred = (p_t >= FROZEN["threshold"]).astype(int)
    tp = int(((pred == 1) & (y_t == 1)).sum()); fn = int(((pred == 0) & (y_t == 1)).sum()); tn = int(((pred == 0) & (y_t == 0)).sum()); fp = int(((pred == 1) & (y_t == 0)).sum())
    res = {"AUC": roc_auc_score(y_t, p_t), "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp), "precision": tp / max(tp + fp, 1), "n_test": len(y_t)}
    return res, (p_t, z_t, y_t)

In [ ]:
# RUN THIS CELL ONCE
test_res, test_scores = final_test_report()
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in test_res.items()})
print("validation sensitivity was", round(sens, 3), "-> test sensitivity", round(test_res["sensitivity"], 3))

### Checkpoint 3 · exactly one test evaluation

In [ ]:
assert len(_test_runs) == 1, f"the test set was used {len(_test_runs)} times; restart the runtime and run the notebook again"
assert abs(test_res["AUC"] - roc_auc_score(test_scores[2], test_scores[0])) < 1e-12
print("Checkpoint 3 passed: the test set was used once")

**Reflection questions.**
1. The validation sensitivity was ≥ 0.90 by construction. What is the test sensitivity? Is the difference larger than the noise? (Next section.)
2. Why is it wrong to look at the test score "just to check" and then change the threshold?
3. A colleague says: "I picked the threshold that maximised accuracy on the test set." Which of the four leakage patterns from this morning is that?

## Plus · put an error bar on every number
**Bootstrap:** resample the images *with replacement*, recompute the metric, repeat 1,000 times, read the 2.5th and 97.5th percentiles.

**TODO (Plus).** Write `bootstrap_ci(metric_fn, scores, labels)`; compute a 95% interval for the validation AUC and for the test AUC (re-using the scores you already have: this is reporting an interval around the one test result, not a second look).

In [ ]:
# [PLUS]
def bootstrap_ci(metric_fn, scores, labels, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n_boot):
        # TODO (Plus): i = random indices with replacement (rng.integers(0, n, n)); skip resamples with only one class;
        # append metric_fn(labels[i], scores[i]) to vals
        pass
    return np.percentile(vals, [2.5, 97.5])

lo, hi = bootstrap_ci(roc_auc_score, p_val, y_val)
print(f"validation AUC {auc_val:.3f}, 95% interval [{lo:.3f}, {hi:.3f}]")
p_t, _, y_t = test_scores
lo_t, hi_t = bootstrap_ci(roc_auc_score, p_t, y_t)
print(f"test AUC       {test_res['AUC']:.3f}, 95% interval [{lo_t:.3f}, {hi_t:.3f}]")

## Plus · can you trust the probability? Calibration
When the model says 0.9, do about 90% of those images really have pneumonia? Group predictions into probability bins and compare the **mean predicted probability** with the **observed fraction** of pneumonia. The **expected calibration error (ECE)** is the weighted average gap.

**TODO (Plus).** Write `ece(probs, labels, n_bins=10)`: for each bin, `(n_bin / N) * |observed fraction - mean probability|`, summed over bins.

In [ ]:
# [PLUS]
def ece(probs, labels, n_bins=10):
    bins = np.minimum((probs * n_bins).astype(int), n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = bins == b
        # TODO (Plus): if the bin is not empty, add  m.mean() * |labels[m].mean() - probs[m].mean()|
        pass
    return float(total)

def reliability(probs, labels, n_bins=10, title=""):
    bins = np.minimum((probs * n_bins).astype(int), n_bins - 1)
    xs = [probs[bins == b].mean() for b in range(n_bins) if (bins == b).any()]
    ys = [labels[bins == b].mean() for b in range(n_bins) if (bins == b).any()]
    plt.plot([0, 1], [0, 1], "k:"); plt.plot(xs, ys, "o-"); plt.xlabel("mean predicted probability"); plt.ylabel("observed fraction of pneumonia")
    plt.title(f"{title} ECE {ece(probs, labels, n_bins):.3f}"); plt.show()

reliability(p_val, y_val, title="validation")
rs = np.random.default_rng(0)
perfect = rs.random(20000); y_perfect = (rs.random(20000) < perfect).astype(int)
over = np.clip(perfect + (perfect - 0.5) * 0.6, 0.001, 0.999)
print("ECE of a perfectly calibrated simulation:", round(ece(perfect, y_perfect), 3), "| of an overconfident one:", round(ece(over, y_perfect), 3))
assert ece(perfect, y_perfect) < 0.02 < ece(over, y_perfect)

**Discuss.** AUC only checks the *order* of the scores; calibration checks their *meaning*. Can a model have an excellent AUC and poor calibration? (Yes: multiply every logit by 3 and the ranking does not change.) Why does it matter if a clinician reads "0.92" as a probability?

## Plus · leakage in three lines (demo, 8 min)
Ten images per patient. Each patient has a fixed "signature" (anatomy, scanner, implants) that a flexible model can memorise. The label is a property of the **patient**, and the true disease signal is weak. We compare two splits with the same model:

- **by image:** shuffle all images, then split (the same patient lands on both sides)
- **by patient:** assign whole patients to a side

**TODO (Plus).** Complete `split_by_patient` so that no patient appears in both train and test (assert).

In [ ]:
# [PLUS]
from sklearn.neighbors import KNeighborsClassifier
rng = np.random.default_rng(0)
n_pat, per_pat, d = 200, 10, 20
sig = rng.normal(0, 1, (n_pat, d))                       # patient signature
pat_label = rng.integers(0, 2, n_pat)                    # label belongs to the patient
disease = rng.normal(0, 1, d)
Xl = np.repeat(sig, per_pat, 0) + 0.5 * rng.normal(0, 1, (n_pat * per_pat, d)) + 0.3 * np.outer(np.repeat(pat_label, per_pat), disease)
yl = np.repeat(pat_label, per_pat); gid = np.repeat(np.arange(n_pat), per_pat)

def split_by_image(n, seed=0):
    idx = np.random.default_rng(seed).permutation(n); k = int(0.8 * n); return idx[:k], idx[k:]

def split_by_patient(groups, seed=0):
    pats = np.random.default_rng(seed).permutation(np.unique(groups)); k = int(0.8 * len(pats))
    # TODO (Plus): train patients = pats[:k]; return the indices of the images of train patients and of the other patients
    return np.array([0]), np.array([1])

def knn_auc(tr, te):
    clf = KNeighborsClassifier(n_neighbors=15).fit(Xl[tr], yl[tr]); return roc_auc_score(yl[te], clf.predict_proba(Xl[te])[:, 1])

tr, te = split_by_image(len(yl)); auc_img = knn_auc(tr, te)
tr2, te2 = split_by_patient(gid); auc_pat = knn_auc(tr2, te2)
assert not (set(gid[tr2]) & set(gid[te2])) and len(tr2) > 100, "a patient appears on both sides (or the split is empty)"
print(f"split by image   : test AUC {auc_img:.3f}  (patients shared: {len(set(gid[tr]) & set(gid[te]))})")
print(f"split by patient : test AUC {auc_pat:.3f}  (patients shared: {len(set(gid[tr2]) & set(gid[te2]))})")

> **Link to the morning.** The by-image score looks wonderful and is a lie: this is scenario 1 of the leakage detective. For PneumoniaMNIST the paper does not say the train/validation split is by patient (see NB05), so treat validation numbers with the same suspicion.

## Challenge · temperature scaling
A single number **T** rescales the logit: `p = sigmoid(z / T)`. It never changes the ranking (so the AUC is identical) but can fix over-confidence. Fit T by minimising the negative log-likelihood on **half of the validation set** and measure ECE on the **other half**. (With real projects you would use a separate calibration set; here the validation set is the only one we may use.)

In [ ]:
# [CHALLENGE]
from scipy.optimize import minimize_scalar
sigmoid = lambda z: 1 / (1 + np.exp(-z))
def nll(T, z, y):
    p = np.clip(sigmoid(z / T), 1e-7, 1 - 1e-7)
    # TODO (Challenge): negative log-likelihood: -mean(y*log(p) + (1-y)*log(1-p))
    return 0.0
def fit_temperature(z, y):
    # TODO (Challenge): minimise nll over T > 0 (hint: optimise log T with minimize_scalar(..., bounds=(-3, 3), method="bounded"))
    return 1.0

zs = rng.normal(0, 2, 20000); ys = (rng.random(20000) < sigmoid(zs)).astype(int); z_over = 3 * zs
T_sim = fit_temperature(z_over, ys)
print(f"simulation: fitted T = {T_sim:.2f} (true value 3.0)")
assert abs(T_sim - 3) < 0.3, "the fitted temperature should recover the factor 3"
perm = np.random.default_rng(1).permutation(len(y_val)); A, B = perm[: len(perm) // 2], perm[len(perm) // 2:]
T = fit_temperature(z_val[A], y_val[A])
print(f"T = {T:.2f} | ECE on half B: {ece(sigmoid(z_val[B]), y_val[B]):.3f} -> {ece(sigmoid(z_val[B] / T), y_val[B]):.3f}")

## Summary
- Report what the task needs (sensitivity, specificity, precision, AUC) **with an interval**, never accuracy alone.
- Choose the threshold from the clinical requirement **on validation**; say the cost out loud.
- The test set is used once. If you look and then change anything, it has become a validation set.
- AUC checks order, calibration checks meaning. A leaky split gives a believable number and a useless tool.

*Next:* NB07 (transfer learning).